In [1]:
import models as mm
import customDatasets
import time
import csv
import matplotlib.pyplot as plt
import json
import torch
import math
import numpy as np
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import json

/Users/Innomius/anaconda3/envs/thesis/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/Users/Innomius/anaconda3/envs/thesis/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c1017RegisterOperatorsD1Ev
  Referenced from: <9AF5A39D-A91D-36CA-BAC2-6A31104C2E9C> /Users/Innomius/anaconda3/envs/thesis/lib/python3.11/site-packages/torchvision/image.so
  Expected in:     <2F21004C-0BB9-30C2-B1C8-B7716FE120B6> /Users/Innomius/anaconda3/envs/thesis/lib/libtorch_cpu.dylib'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
# Local data location
test_indices = [0]
camera_imgs_dir_path = "../images/camera"
microscope_imgs_dir_path = "../images/microscope"
labels_file_path = "./labelsinfo.csv"
label_type = "om-regression"
model_weights_path = "../weights/2b"
dummy_camera_model_weights_path = "../weights/camera"
dummy_microscope_model_weights_path = "../weights/microscope.zip"

# Define model and training configurations
microscope_img_roi = 1920
camera_img_roi = 1000
img_divisions_n = 2

microscope_img_channels = 3
microscope_height_resize = 224
microscope_width_resize = 224

camera_img_channels = 3
camera_height_resize = 224
camera_width_resize = 224

rotations = [0]

# Loaders batch size
batch_size = 16

# Models Params
model_output_dims = {
    "om-regression": 4,
    "mineral-regression": 3
}
model_type = "tiny"
pre_trained = False
train_only_last_layer = False

# Build cropboxes
microscope_sub_img_dim = microscope_img_roi // img_divisions_n
camera_sub_img_dim = camera_img_roi // img_divisions_n

microscope_img_left = 0
microscope_img_upper = 0
microscope_img_right = microscope_sub_img_dim
microscope_img_lower = microscope_sub_img_dim

camera_img_left = 0
camera_img_upper = 0
camera_img_right = camera_sub_img_dim
camera_img_lower = camera_sub_img_dim

# Select one quadrant per picture
microscope_cropboxes = [(microscope_img_left, microscope_img_upper, microscope_img_right, microscope_img_lower)] # (left, upper, right, lower)
camera_cropboxes = [(camera_img_left, camera_img_upper, camera_img_right, camera_img_lower)] # (left, upper, right, lower)
        
# Select all 4 quadrants per picture
# microscope_cropboxes = customDatasets.createCropBoxes(img_divisions_n, microscope_sub_img_dim)
# camera_cropboxes = customDatasets.createCropBoxes(img_divisions_n, camera_sub_img_dim)

# Transforms
custome_normalization_transform = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

test_dataset = customDatasets.TwoImagesCropboxRotationDataset(
    camera_imgs_dir_path,
    microscope_imgs_dir_path,
    labels_file_path,
    label_type,
    test_indices,
    camera_cropboxes,
    microscope_cropboxes,
    camera_img_channels,
    microscope_img_channels,
    camera_height_resize,
    camera_width_resize,
    microscope_height_resize,
    microscope_width_resize,
    rotations_values=rotations,
    camera_transform=custome_normalization_transform,
    micro_transform=custome_normalization_transform
)

test_dataset_n = test_dataset.__len__()

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

# Build model
model = mm.ConvNext2BTrainer.instantiate_model(_, model_type, label_type, dummy_microscope_model_weights_path, dummy_camera_model_weights_path, train_only_last_layer)

model.load_state_dict(torch.load(model_weights_path, map_location=torch.device('cpu')))
model.eval()

batch_counter = 0
total_iterations = math.ceil(test_dataset_n / batch_size)

test_labels = []
test_raw_preds = []

for camera_inputs, micro_inputs, labels in test_loader:
    batch_counter += 1
    print("Batch: {}/{}".format(batch_counter, total_iterations))

    # Make predictions for this batch
    raw_preds = model(micro_inputs, camera_inputs)

    # Save predictions and labels
    test_labels.append(labels.numpy())
    test_raw_preds.append(raw_preds.detach().numpy())

predictions = np.concatenate(test_raw_preds, axis=0)
labels = np.concatenate(test_labels, axis=0)

0 samples processed
Batch: 1/1


In [3]:
predictions

array([[0.6709235 , 0.2098832 , 0.07819568, 0.04099761]], dtype=float32)

In [6]:
mse = np.mean((labels - predictions) ** 2, axis=0)
print("Per feature MSE:", mse)

overall_mse = np.mean(mse)
print("Overall MSE:", overall_mse)

Per feature MSE: [8.5276872e-07 6.8208744e-04 1.1019527e-03 8.1042985e-05]
Overall MSE: 0.000466484
